# Modellierung mit Prophet — PM2.5 Beijing

**Capstone-Projekt · CRISP-DM Phase 4 (Modeling)**  
Bettina Gertjerenken · Kai Steffen

In diesem Notebook trainieren wir **Facebook Prophet** auf den in Kapitel 03 vorbereiteten Daten — 
**einmal mit den Basis-Daten** und **einmal mit den behandelten Daten** — und vergleichen die Genauigkeit über **MAE** und **RMSE**.

**Ablauf:** Daten laden → Prophet trainieren (3 Jahre) → das Testjahr vorhersagen → mit den echten Werten vergleichen → schöne Grafiken.

> ⚙️ **Kernel:** Dieses Notebook braucht die Umgebung **`ts-tutorial`** (Prophet ist dort installiert). 
Oben rechts als Kernel **„Python (ts-tutorial)"** wählen.

> 📁 **Voraussetzung:** Die CSVs aus Kapitel 03 müssen existieren — also dort einmal „Run All" ausführen, 
damit `../data/prepared/basis/` und `../data/prepared/behandelt/` gefüllt sind.

## 1. Bibliotheken laden

In [ ]:
import warnings; warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import seaborn as sns

from prophet import Prophet



sns.set_theme(style="whitegrid")

plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 13, "axes.titleweight": "bold"})



# Farbpalette (wie in den anderen Kapiteln)

AMBER, RUST, SLATE, TEAL = "#E0912F", "#C4471C", "#3A4148", "#1C7293"



STATION = "Aotizhongxin"     # gleiche Beispielstation wie in Kapitel 03

PREP = Path("../data/prepared")

## 2. Hilfsfunktionen: Laden & Metriken

Wir laden Training/Test je Variante und definieren **MAE** und **RMSE** — die zwei Standard-Fehlermaße aus der Literatur. 
Beide messen den durchschnittlichen Vorhersagefehler in µg/m³; RMSE bestraft große Ausreißer stärker.

In [ ]:
def lade(variante, station=STATION):

    """variante = 'basis' oder 'behandelt' -> (train, test) DataFrames mit ds, y, ..."""

    tr = pd.read_csv(PREP / variante / f"prophet_train_{station}.csv", parse_dates=["ds"])

    te = pd.read_csv(PREP / variante / f"prophet_test_{station}.csv",  parse_dates=["ds"])

    return tr, te



def mae(y_true, y_pred):

    return float(np.mean(np.abs(y_true - y_pred)))



def rmse(y_true, y_pred):

    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

## 3. Prophet trainieren & vorhersagen (eine Funktion für beide Varianten)

Damit der Vergleich fair ist, nutzen wir **dieselbe Funktion** für Basis und behandelt. 
Wir starten **univariat** (nur `ds` + `y`) — Prophet erkennt Trend, Jahres-, Wochen- und Tagesmuster automatisch. 
Die Vorhersage machen wir für die Zeitstempel des Testjahres.

In [ ]:
def fit_predict(train, test):

    """Trainiert Prophet auf train (ds,y) und sagt die ds aus test vorher."""

    m = Prophet(

        yearly_seasonality=True,

        weekly_seasonality=True,

        daily_seasonality=True,

        changepoint_prior_scale=0.05,   # Standard; groesser = flexiblerer Trend

    )

    m.fit(train[["ds", "y"]])                    # <- Training (3 Jahre)

    forecast = m.predict(test[["ds"]])           # <- Vorhersage fuer Testjahr

    ergebnis = test[["ds", "y"]].merge(

        forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]], on="ds")

    ergebnis["yhat"] = ergebnis["yhat"].clip(lower=0)   # PM2.5 kann nicht negativ sein

    return m, forecast, ergebnis

## 4. Schöne Plot-Funktionen

Statt der Standard-Prophet-Plots bauen wir zwei aufgeräumte Grafiken: (a) Vorhersage vs. echte Werte im Testjahr 
mit Unsicherheitsband, und (b) eine kompakte Fehler-Übersicht.

In [ ]:
def plot_forecast(ergebnis, titel, farbe=TEAL, tage_glatt=1):

    """Vorhersage vs. echte Werte im Testjahr (auf Tagesmittel geglaettet fuer Lesbarkeit)."""

    g = ergebnis.set_index("ds").resample(f"{tage_glatt}D").mean()

    plt.figure(figsize=(12, 4))

    plt.fill_between(g.index, g["yhat_lower"], g["yhat_upper"], color=farbe, alpha=0.18,

                     label="Prophet-Unsicherheit")

    plt.plot(g.index, g["y"],    color=SLATE, lw=1.4, label="echte Werte")

    plt.plot(g.index, g["yhat"], color=farbe, lw=1.6, label="Prophet-Vorhersage")

    plt.ylabel("PM2.5 (µg/m³)"); plt.title(titel); plt.legend(loc="upper right"); plt.tight_layout()

    plt.show()



def plot_scatter(ergebnis, titel, farbe=TEAL):

    """Streudiagramm echte vs. vorhergesagte Werte."""

    plt.figure(figsize=(4.6, 4.6))

    plt.scatter(ergebnis["y"], ergebnis["yhat"], s=4, alpha=0.2, color=farbe)

    lim = max(ergebnis["y"].max(), ergebnis["yhat"].max())

    plt.plot([0, lim], [0, lim], color=RUST, lw=1.2, ls="--", label="perfekt")

    plt.xlabel("echte PM2.5"); plt.ylabel("Vorhersage"); plt.title(titel)

    plt.legend(); plt.tight_layout(); plt.show()

## 5. Lauf A — Basis-Daten

Zuerst die **unbehandelten** Daten. Hinweis: Das Training auf ~26.000 Stundenwerten dauert je nach Rechner **1–3 Minuten**.

In [ ]:
train_b, test_b = lade("basis")

print(f"Basis: Training {len(train_b)} Zeilen, Test {len(test_b)} Zeilen")



m_b, fc_b, res_b = fit_predict(train_b, test_b)



mae_b, rmse_b = mae(res_b["y"], res_b["yhat"]), rmse(res_b["y"], res_b["yhat"])

print(f"Basis  ->  MAE = {mae_b:.2f}   RMSE = {rmse_b:.2f}")

In [ ]:
plot_forecast(res_b, f"Basis — Prophet-Vorhersage Testjahr ({STATION})", farbe=RUST)

In [ ]:
plot_scatter(res_b, "Basis — echt vs. Vorhersage", farbe=RUST)

**Prophet-Komponenten (Basis):** Trend, Jahres-, Wochen- und Tagesverlauf, die Prophet gelernt hat. 
Hier sieht man z. B. den Winter-Peak und den Tagesgang aus Kapitel 02 wieder.

In [ ]:
fig = m_b.plot_components(fc_b); plt.show()

## 6. Lauf B — Behandelte Daten

Jetzt die **behandelten** Daten (lange Lücken entfernt, Ausreißer gekappt). Gleiche Funktion, gleicher Ablauf.

In [ ]:
train_t, test_t = lade("behandelt")

print(f"Behandelt: Training {len(train_t)} Zeilen, Test {len(test_t)} Zeilen")



m_t, fc_t, res_t = fit_predict(train_t, test_t)



mae_t, rmse_t = mae(res_t["y"], res_t["yhat"]), rmse(res_t["y"], res_t["yhat"])

print(f"Behandelt  ->  MAE = {mae_t:.2f}   RMSE = {rmse_t:.2f}")

In [ ]:
plot_forecast(res_t, f"Behandelt — Prophet-Vorhersage Testjahr ({STATION})", farbe=TEAL)

In [ ]:
plot_scatter(res_t, "Behandelt — echt vs. Vorhersage", farbe=TEAL)

In [ ]:
fig = m_t.plot_components(fc_t); plt.show()

## 7. Vergleich Basis vs. Behandelt

Die entscheidende Frage: **Bringt die Datenbehandlung eine bessere Vorhersage?** 
Wir stellen MAE und RMSE beider Läufe gegenüber.

> ⚠️ **Fairer Vergleich:** Die behandelten Testdaten sind leicht anders (gekappt, ohne lange Lücken). 
Für einen ganz sauberen Vergleich könnte man beide auf **denselben Zeitstempeln** auswerten — 
hier reicht der direkte Vergleich der Fehlermaße als erste Orientierung.

In [ ]:
vergleich = pd.DataFrame({

    "Variante": ["Basis", "Behandelt"],

    "MAE":  [mae_b, mae_t],

    "RMSE": [rmse_b, rmse_t],

})

print(vergleich.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))

for i, (metrik, werte) in enumerate([("MAE", [mae_b, mae_t]), ("RMSE", [rmse_b, rmse_t])]):

    bars = ax[i].bar(["Basis", "Behandelt"], werte, color=[RUST, TEAL])

    ax[i].set_title(metrik); ax[i].set_ylabel("µg/m³")

    for b, v in zip(bars, werte):

        ax[i].text(b.get_x()+b.get_width()/2, v, f"{v:.1f}", ha="center", va="bottom")

plt.suptitle(f"Prophet — Basis vs. Behandelt ({STATION})", fontweight="bold")

plt.tight_layout(); plt.show()

## 8. Optional: Wetter-Regressoren & Log-Ziel

Zwei einfache Erweiterungen, die ihr ausprobieren könnt (aus der Aufbereitung liegen die Spalten bereit):

**a) Wetter-Regressoren** — Prophet zusätzliche Einflussgrößen geben:
```python
REGRESSOREN = ["TEMP", "DEWP", "PRES", "WSPM", "RAIN", "wd_sin", "wd_cos"]
m = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=True)
for r in REGRESSOREN:
    m.add_regressor(r)
m.fit(train[["ds", "y"] + REGRESSOREN])
forecast = m.predict(test[["ds"] + REGRESSOREN])   # Regressoren im Test noetig
```

**b) Log-Ziel** (nur behandelte Daten haben `y_log`) — gegen die Rechtsschiefe:
```python
tr = train.rename(columns={"y": "y_raw", "y_log": "y"})   # auf log-Skala trainieren
m = Prophet(...); m.fit(tr[["ds", "y"]])
fc = m.predict(test[["ds"]])
fc["yhat"] = np.expm1(fc["yhat"])            # zurueck auf µg/m³ transformieren
```

## 9. Ergebnis & Ausblick

**Ergebnis:** Prophet ist auf beiden Datenvarianten trainiert und auf dem Testjahr ausgewertet — mit MAE/RMSE und Grafiken. 
Der direkte Vergleich zeigt, ob die Datenbehandlung aus Kapitel 03 die Vorhersage verbessert.

**Nächste Schritte:**
- Erweiterung auf **alle 12 Stationen** (Schleife über `lade(variante, station)` und Mittelung der Fehlermaße).
- Dieselbe Test-Logik für **Chronos** und **PatchTST** — dann sind alle drei Modelle fair vergleichbar (gleicher Split, gleiche Metriken).

Damit ist die Prophet-Modellierung abgeschlossen; der Modellvergleich folgt im nächsten Schritt.